# NeuroSleep — Notebook 04: Deep Learning with TensorFlow/Keras

**Project**: Automated sleep stage classification from EEG signals using deep learning.

**Author**: Marcos Vinícius Rocha Gomes

**Date**: May 2026

---

## Objectives of this notebook

1. Load the raw normalized epochs produced in Notebook 02.
2. Design and train a **1D Convolutional Neural Network** in TensorFlow/Keras to classify sleep stages directly from raw EEG signals.
3. Use the same **subject-aware split** as the baselines for a fair comparison.
4. Apply class weights, early stopping, and learning rate scheduling.
5. Evaluate with the same metrics (macro-F1, balanced accuracy, Cohen's kappa).
6. Compare against the Random Forest / SVM baselines from Notebook 03.
7. Save the trained model and final metrics.

## Why a 1D CNN for sleep EEG?

Sleep stages have characteristic **spectro-temporal patterns** in EEG:
- **Sleep spindles** (~12–14 Hz bursts of ~1s duration in N2)
- **K-complexes** (sharp negative-positive waveforms in N2)
- **Delta waves** (slow, high-amplitude oscillations in N3)
- **Alpha rhythm** (8–13 Hz dominant when relaxed but awake)

These are **local patterns in time**, which is exactly what 1D convolutions are designed to detect. Each conv layer learns increasingly abstract waveform detectors; pooling provides translation invariance (a spindle is a spindle regardless of where in the 30-s epoch it appears).

## Why this matters for the comparison

The CNN receives the **raw signal** — no feature engineering. If it beats the Random Forest baseline that uses 80+ hand-crafted features, that's evidence the network is learning representations a human didn't think to extract. If it doesn't, the honest conclusion is that classical features already capture what matters.

## 1. Imports and configuration

In [ ]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks, regularizers

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    balanced_accuracy_score,
    cohen_kappa_score,
    accuracy_score,
)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 100

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

STAGE_ORDER = ["W", "N1", "N2", "N3", "REM"]

PROCESSED_DIR = Path("../data/processed")
RESULTS_DIR = Path("../data/results")
MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version:      {keras.__version__}")
print(f"GPU available:      {len(tf.config.list_physical_devices('GPU')) > 0}")

## 2. Load preprocessed signals

Notebook 02 saved the normalized 30-s epochs in NumPy archive format. Shape: `(n_epochs, n_channels, n_samples)`.

In [ ]:
data = np.load(PROCESSED_DIR / "epochs.npz", allow_pickle=True)
X = data["X"].astype(np.float32)
y_str = data["y"]
groups = data["subjects"]

print(f"X shape: {X.shape}")
print(f"  (n_epochs={X.shape[0]}, n_channels={X.shape[1]}, n_samples_per_epoch={X.shape[2]})")
print(f"y shape: {y_str.shape}")
print(f"Unique stages: {np.unique(y_str)}")
print(f"Unique subjects: {np.unique(groups)}")

Keras expects the channel dimension **last** for 1D convolutions: `(n_epochs, n_samples, n_channels)`. We also encode string labels to integers.

In [ ]:
# Transpose: (n, ch, t) -> (n, t, ch)
X = np.transpose(X, (0, 2, 1))

# Encode labels, fixed order so it matches STAGE_ORDER
le = LabelEncoder()
le.fit(STAGE_ORDER)
y = le.transform(y_str)
n_classes = len(le.classes_)

print(f"X final shape (Keras format): {X.shape}")
print(f"y encoded shape: {y.shape}")
print(f"Classes ({n_classes}): {list(le.classes_)}")

## 3. Subject-aware split (same scheme as Notebook 03)

We split into train / validation / test by subject. The test set must contain subjects never seen during training OR hyperparameter selection.

In [ ]:
# First split: trainval vs test (~70/30 by subjects)
outer = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=RANDOM_STATE)
trainval_idx, test_idx = next(outer.split(X, y, groups=groups))

X_trainval, X_test = X[trainval_idx], X[test_idx]
y_trainval, y_test = y[trainval_idx], y[test_idx]
groups_trainval = groups[trainval_idx]
groups_test = groups[test_idx]

# Second split: train vs val inside trainval (by subjects again)
n_subj_trainval = len(np.unique(groups_trainval))
if n_subj_trainval >= 2:
    inner = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
    train_idx, val_idx = next(inner.split(X_trainval, y_trainval, groups=groups_trainval))
    X_train, X_val = X_trainval[train_idx], X_trainval[val_idx]
    y_train, y_val = y_trainval[train_idx], y_trainval[val_idx]
    groups_train = groups_trainval[train_idx]
    groups_val = groups_trainval[val_idx]
else:
    # Fallback if too few subjects: random epoch split within the only trainval subject (not ideal)
    print("WARNING: only one subject in trainval. Falling back to random epoch split.")
    rng = np.random.RandomState(RANDOM_STATE)
    perm = rng.permutation(len(X_trainval))
    cut = int(0.8 * len(perm))
    train_idx, val_idx = perm[:cut], perm[cut:]
    X_train, X_val = X_trainval[train_idx], X_trainval[val_idx]
    y_train, y_val = y_trainval[train_idx], y_trainval[val_idx]
    groups_train = groups_trainval[train_idx]
    groups_val = groups_trainval[val_idx]

print(f"Train: {X_train.shape[0]:,} epochs, subjects {sorted(np.unique(groups_train).tolist())}")
print(f"Val:   {X_val.shape[0]:,} epochs, subjects {sorted(np.unique(groups_val).tolist())}")
print(f"Test:  {X_test.shape[0]:,} epochs, subjects {sorted(np.unique(groups_test).tolist())}")

# Sanity
train_subjects = set(np.unique(groups_train).tolist())
val_subjects = set(np.unique(groups_val).tolist())
test_subjects = set(np.unique(groups_test).tolist())
assert train_subjects.isdisjoint(val_subjects), "Train/val subject leakage!"
assert train_subjects.isdisjoint(test_subjects), "Train/test subject leakage!"
assert val_subjects.isdisjoint(test_subjects), "Val/test subject leakage!"
print("\n✓ No subject overlap across train/val/test.")

## 4. Class weights for imbalanced training

Same logic as Notebook 03: compute weights inversely proportional to class frequency.

In [ ]:
classes_present = np.unique(y_train)
weights = compute_class_weight("balanced", classes=classes_present, y=y_train)
class_weight_dict = dict(zip(classes_present.tolist(), weights.tolist()))

print("Class weights (label index -> weight):")
for cls_idx, w in class_weight_dict.items():
    print(f"  {cls_idx} ({le.inverse_transform([cls_idx])[0]:>3}): {w:.3f}")

## 5. Model architecture

A compact, well-justified 1D CNN. Each block does:

1. **Conv1D**: learns local waveform detectors at a given temporal scale.
2. **BatchNormalization**: stabilizes activations, allows higher learning rates.
3. **ReLU activation** (implicit in Conv1D via `activation`).
4. **MaxPooling1D**: gradually reduces temporal resolution while keeping the strongest activations.
5. **Dropout**: regularization against subject-specific overfitting.

We stack three blocks with increasing channel widths (32 → 64 → 128), then global average pooling + dense classifier. The receptive field after three pools (factor 4 each) covers several seconds of signal — enough to capture spindles, K-complexes, and slow waves.

L2 regularization on dense layer further fights overfitting given few subjects.

In [ ]:
def build_cnn(input_shape: tuple, n_classes: int) -> keras.Model:
    """1D CNN for sleep stage classification from raw EEG."""
    inputs = keras.Input(shape=input_shape, name="eeg_epoch")

    # Block 1: short receptive field for fast oscillations
    x = layers.Conv1D(32, kernel_size=11, padding="same", activation="relu")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(pool_size=4)(x)
    x = layers.Dropout(0.3)(x)

    # Block 2: medium receptive field
    x = layers.Conv1D(64, kernel_size=11, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(pool_size=4)(x)
    x = layers.Dropout(0.3)(x)

    # Block 3: longer receptive field for slow rhythms
    x = layers.Conv1D(128, kernel_size=11, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(pool_size=4)(x)
    x = layers.Dropout(0.4)(x)

    # Global pooling -> classifier
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(
        64, activation="relu",
        kernel_regularizer=regularizers.l2(1e-4),
    )(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(n_classes, activation="softmax")(x)

    model = keras.Model(inputs=inputs, outputs=outputs, name="NeuroSleepCNN")
    return model


input_shape = X_train.shape[1:]  # (n_samples, n_channels)
model = build_cnn(input_shape=input_shape, n_classes=n_classes)
model.summary()

## 6. Compile the model

- **Loss**: sparse categorical crossentropy (integer labels, multi-class).
- **Optimizer**: Adam with default LR (`1e-3`); the scheduler will reduce it on plateau.
- **Tracked metrics**: accuracy (cheap) and a custom callback for macro-F1 (the metric that matters).

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

## 7. Callbacks

Standard set: early stopping on validation loss, learning rate reduction on plateau, model checkpoint of best weights, and a custom macro-F1 logger.

In [ ]:
class MacroF1Callback(callbacks.Callback):
    """Logs macro-F1 on the validation set at the end of each epoch."""

    def __init__(self, X_val, y_val):
        super().__init__()
        self.X_val = X_val
        self.y_val = y_val
        self.history = []

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        y_pred = self.model.predict(self.X_val, verbose=0).argmax(axis=1)
        f1 = f1_score(self.y_val, y_pred, average="macro", zero_division=0)
        logs["val_macro_f1"] = f1
        self.history.append(f1)
        print(f"  → val_macro_f1: {f1:.4f}")


checkpoint_path = MODELS_DIR / "cnn_best.keras"

f1_cb = MacroF1Callback(X_val, y_val)

cb_list = [
    callbacks.EarlyStopping(
        monitor="val_loss",
        patience=8,
        restore_best_weights=True,
        verbose=1,
    ),
    callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=4,
        min_lr=1e-6,
        verbose=1,
    ),
    callbacks.ModelCheckpoint(
        filepath=str(checkpoint_path),
        monitor="val_loss",
        save_best_only=True,
        verbose=0,
    ),
    f1_cb,
]

## 8. Train

In [ ]:
EPOCHS = 50
BATCH_SIZE = 64

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weight_dict,
    callbacks=cb_list,
    verbose=2,
)

## 9. Training curves

Diagnostic plots: are we overfitting? Did the LR scheduler help?

In [ ]:
hist = history.history
epochs_ran = range(1, len(hist["loss"]) + 1)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(epochs_ran, hist["loss"], label="Train", linewidth=2)
axes[0].plot(epochs_ran, hist["val_loss"], label="Val", linewidth=2)
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Sparse categorical crossentropy")
axes[0].legend()

axes[1].plot(epochs_ran, hist["accuracy"], label="Train", linewidth=2)
axes[1].plot(epochs_ran, hist["val_accuracy"], label="Val", linewidth=2)
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()

axes[2].plot(epochs_ran, f1_cb.history, label="Val macro-F1", linewidth=2, color="#27ae60")
axes[2].set_title("Validation macro-F1")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("Macro F1")
axes[2].legend()

plt.tight_layout()
plt.show()

### What to look for

- **Healthy training**: both losses decreasing, val loss tracking train loss within a reasonable gap.
- **Overfitting signal**: train loss plummets while val loss flattens or rises — early stopping should have caught it.
- **Underfitting signal**: both curves still descending sharply at the end — you may want more epochs or a larger model.

## 10. Evaluate on the held-out test set

Final, single shot. No more decisions after this number.

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, batch_size=BATCH_SIZE, verbose=0)
y_pred = model.predict(X_test, batch_size=BATCH_SIZE, verbose=0).argmax(axis=1)

cnn_metrics = {
    "model": "CNN_1D_Keras",
    "accuracy": float(accuracy_score(y_test, y_pred)),
    "balanced_accuracy": float(balanced_accuracy_score(y_test, y_pred)),
    "macro_f1": float(f1_score(y_test, y_pred, average="macro", zero_division=0)),
    "weighted_f1": float(f1_score(y_test, y_pred, average="weighted", zero_division=0)),
    "cohen_kappa": float(cohen_kappa_score(y_test, y_pred)),
    "test_loss": float(test_loss),
    "epochs_trained": int(len(hist["loss"])),
}

print("=== CNN (1D, Keras) — test set ===")
for k, v in cnn_metrics.items():
    if isinstance(v, float):
        print(f"  {k:18s} {v:.4f}")
    else:
        print(f"  {k:18s} {v}")

In [ ]:
print("CNN — per-class classification report:\n")
print(classification_report(
    y_test, y_pred,
    target_names=le.classes_,
    digits=3, zero_division=0,
))

## 11. Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred, normalize="true")

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt=".2f", cmap="Blues",
    xticklabels=le.classes_, yticklabels=le.classes_,
    cbar=False, vmin=0, vmax=1, ax=ax,
)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("CNN (1D Keras) — normalized confusion matrix")
plt.tight_layout()
plt.show()

## 12. Compare against the baselines from Notebook 03

Load the baseline metrics and stack them with the CNN results.

In [ ]:
baseline_path = RESULTS_DIR / "baseline_metrics.json"

if baseline_path.exists():
    with open(baseline_path) as f:
        baseline_results = json.load(f)
    rf_metrics = baseline_results["random_forest"]
    svm_metrics = baseline_results["svm_rbf"]

    rows = []
    for m in [rf_metrics, svm_metrics, cnn_metrics]:
        rows.append({
            "model": m["model"],
            "accuracy": m["accuracy"],
            "balanced_accuracy": m["balanced_accuracy"],
            "macro_f1": m["macro_f1"],
            "weighted_f1": m["weighted_f1"],
            "cohen_kappa": m["cohen_kappa"],
        })
    comparison = pd.DataFrame(rows).set_index("model")
    print("Comparison across models:\n")
    display(comparison.round(4))
else:
    print("Baseline file not found. Run Notebook 03 first.")
    comparison = pd.DataFrame([cnn_metrics]).set_index("model")

In [ ]:
metric_cols = ["accuracy", "balanced_accuracy", "macro_f1", "weighted_f1", "cohen_kappa"]
plot_df = comparison[metric_cols].reset_index().melt(
    id_vars="model", var_name="metric", value_name="score"
)

fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(data=plot_df, x="metric", y="score", hue="model", palette="Set2", ax=ax)
ax.set_title("All models — test set comparison")
ax.set_ylabel("Score")
ax.set_ylim(0, 1)
ax.legend(loc="lower right")
for container in ax.containers:
    ax.bar_label(container, fmt="%.3f", fontsize=8, padding=2)
plt.tight_layout()
plt.show()

## 13. Persist the model and final results

In [ ]:
# Save final model (in addition to best checkpoint saved by callback)
final_model_path = MODELS_DIR / "cnn_final.keras"
model.save(final_model_path)
print(f"Saved final model to {final_model_path}")

# Save metrics
cnn_results_path = RESULTS_DIR / "cnn_metrics.json"
with open(cnn_results_path, "w") as f:
    json.dump({
        "cnn": cnn_metrics,
        "train_subjects": sorted([int(s) for s in np.unique(groups_train)]),
        "val_subjects": sorted([int(s) for s in np.unique(groups_val)]),
        "test_subjects": sorted([int(s) for s in np.unique(groups_test)]),
        "hyperparameters": {
            "epochs": EPOCHS,
            "batch_size": BATCH_SIZE,
            "optimizer": "adam",
            "initial_lr": 1e-3,
        },
    }, f, indent=2)
print(f"Saved CNN metrics to {cnn_results_path}")

# Save predictions
cnn_preds_path = RESULTS_DIR / "cnn_predictions.parquet"
pd.DataFrame({
    "subject": groups_test,
    "true": le.inverse_transform(y_test),
    "cnn_pred": le.inverse_transform(y_pred),
}).to_parquet(cnn_preds_path, index=False)
print(f"Saved CNN predictions to {cnn_preds_path}")

## 14. Summary and honest conclusions

### What we built

- A 1D CNN in TensorFlow/Keras (~3 conv blocks → global average pooling → dense classifier).
- Trained from **raw normalized signals** — no hand-crafted features.
- Same subject-aware split as the baselines, ensuring an apples-to-apples comparison.
- Standard training discipline: class weights, early stopping, LR reduction on plateau, custom macro-F1 logging, best-model checkpointing.

### What the comparison tells us

Three honest scenarios when interpreting the chart:

1. **CNN wins clearly on macro-F1 and balanced accuracy**: the network learned representations beyond what classical features capture. This is the publication-quality outcome.
2. **CNN ties or barely wins**: features were already good; the network's complexity isn't justified for this data scale.
3. **CNN underperforms**: most likely cause is too few subjects — DL is data-hungry. Increasing subjects in Notebook 02 is the first remedy.

Report whichever scenario you actually observe. Both are scientifically valid.

### Limitations of this notebook

- **Epoch independence assumption**: we still classify each 30-s window in isolation. Real sleep scoring uses temporal context — adding an LSTM/Transformer over CNN features (sequence-to-sequence) typically adds 3–8 macro-F1 points. Out of scope here, listed as future work.
- **No hyperparameter search**: kernel size, depth, dropout — all picked by reasoning, not by random/Bayesian search.
- **Single channel modeling only when n_channels=2**: we use both EEG channels but treat them as features to the same conv, not as separate streams.

### Next steps (Notebook 05 — persistence)

- Push all results (baselines + CNN) into the PostgreSQL schema.
- Log the full experiment (hyperparameters + metrics + artifact paths) via MLflow.
- Wrap the inference pipeline in a small CLI for the Docker container.